In [5]:
#Define dir
dir = "/Users/felipesanchez/Documents/UoM/PhD/Phd Third Year/Chapter Fertility/ModellingF/"

using Turing, MCMCChains, Distributions, RData, CSV, DataFrames, LinearAlgebra, StatsFuns, Random, Serialization

#Importing the exposures (in thousands) and birth counts by education
ef1 = CSV.read(dir * "Data_Created/exfe1.csv", DataFrame) .* 1000
ef2 = CSV.read(dir * "Data_Created/exfe2.csv", DataFrame) .* 1000
ef3 = CSV.read(dir * "Data_Created/exfe3.csv", DataFrame) .* 1000
ef4 = CSV.read(dir * "Data_Created/exfe4.csv", DataFrame) .* 1000

b1 = CSV.read(dir * "Data_Created/fx1.csv", DataFrame)
b2 = CSV.read(dir * "Data_Created/fx2.csv", DataFrame)
b3 = CSV.read(dir * "Data_Created/fx3.csv", DataFrame)
b4 = CSV.read(dir * "Data_Created/fx4.csv", DataFrame)

# The log fertility (lmu) is calculated as the natural logarithm of the ratio of two datasets (e.g., b1/ef1).
#Females
lmuf1 = round.(log.(b1 ./ ef1), digits=6)
lmuf2 = round.(log.(b2 ./ ef2), digits=6)
lmuf3 = round.(log.(b3 ./ ef3), digits=6)
lmuf4 = round.(log.(b4 ./ ef4), digits=6)

# Calculating log exposures for  females
# This is done by applying the natural logarithm to each column in the datasets and rounding to six decimal places.
#Females
logef1 = mapcols(c -> round.(log.(c), digits=6), ef1)
logef2 = mapcols(c -> round.(log.(c), digits=6), ef2)
logef3 = mapcols(c -> round.(log.(c), digits=6), ef3)
logef4 = mapcols(c -> round.(log.(c), digits=6), ef4)

# 'b' array: Combines fertility datasets.
b = [b1, b2, b3, b4]

# 'e' array: Combines population count datasets (exposure data) for females.
e = [ef1, ef2, ef3, ef4]

# 'loge' array: Contains the natural logarithm of the population counts for analytical purposes.
loge = [logef1, logef2, logef3, logef4]

# 'lmu' array: Combines log fertility rates.
lmu = [lmuf1, lmuf2, lmuf3, lmuf4]

#Serialize
serialize(dir * "Data_Created/b.jls", b)
serialize(dir * "Data_Created/loge.jls", loge)
serialize(dir * "Data_Created/lmu.jls", lmu)
# Deserialize
#b = deserialize(dir * "Data_Created/b.jls")
#loge = deserialize(dir * "Data_Created/loge.jls")


In [7]:
@model function model1(bir, loge, h)
    # Verify that all matrices have the same dimensions
    if size(bir) != size(loge)
        error("The sizes of the matrices are different")
    else
        n_ages, n_periods = size(bir)
        n_for = h
        #n_groups = length(bir)
        # Prior for a 
        a ~ MvNormal(zeros(n_ages), 10)
        # Priors for b
        b ~ Dirichlet(ones(n_ages))
        # Priors and random walk for k
        sigma ~ Exponential(0.1)
        #Drift RW
        c ~ Normal(0, sqrt(10))
        #Init k vectors
        k = Vector(undef, n_periods)
        #Prior for k
        k[1] ~ Normal(c, sigma)
        #Generate Random Walk
        for i in 2:n_periods
            k[i] ~ Normal(c + k[i-1], sigma)
        end
        #Normalizing k (restriction sum k is zero)
        sum_k = sum(k)
        for i in 1:n_periods
            k[i] = k[i] - sum_k / n_periods
        end
        #Priors to Log mortality
        Sigma ~ Exponential(1)
        #Init Matrix to dead count
        bir_est = Matrix(undef, n_ages, n_periods)
        #Init matrix to log mortality (Quantiles generated)
        logmu = Matrix(undef, n_ages, n_periods)
        
        # Likelihood for d1, d2, d3 and d4
        ###################################
        for x in 1:n_ages
            for t in 1:n_periods
                logmu[x, t] ~ Normal(a[x] + b[x] * k[t], Sigma)
                bir_est[x, t] = exp(logmu[x, t] + loge[x, t])
                bir[x, t] ~ Poisson(bir_est[x, t])
            end
        end
        #Generated quantities (Forecast and validation)
        ###############################################
        #Forecasting kappa
        kf= Vector(undef, n_for)
        
        kf[1] ~ Normal(c+ k[n_periods], sigma)
        
        # Generate kf values
        for t in 2:n_for
            kf[t] ~ Normal(c + kf[t - 1], sigma)
        end
    end
end 

model1 (generic function with 2 methods)

In [11]:
modelesp1f = model1(b1, logef1, 5)
chain_model1f = sample(modelesp1f, NUTS(0.65), 10)

┌ Info: Found initial step size
└   ϵ = 1.9073486328125e-7
Sampling: 100%|█████████████████████████████████████████| Time: 0:00:01


Chains MCMC chain (10×179×1 Array{Float64, 3}):

Iterations        = 6:1:15
Number of chains  = 1
Samples per chain = 10
Wall duration     = 11.64 seconds
Compute duration  = 11.64 seconds
parameters        = a[1], a[2], a[3], a[4], a[5], a[6], b[1], b[2], b[3], b[4], b[5], b[6], sigma, c, k[1], k[2], k[3], k[4], k[5], k[6], k[7], k[8], k[9], k[10], k[11], k[12], k[13], k[14], k[15], k[16], k[17], k[18], k[19], k[20], k[21], Sigma, logmu[1,1], logmu[1,2], logmu[1,3], logmu[1,4], logmu[1,5], logmu[1,6], logmu[1,7], logmu[1,8], logmu[1,9], logmu[1,10], logmu[1,11], logmu[1,12], logmu[1,13], logmu[1,14], logmu[1,15], logmu[1,16], logmu[1,17], logmu[1,18], logmu[1,19], logmu[1,20], logmu[1,21], logmu[2,1], logmu[2,2], logmu[2,3], logmu[2,4], logmu[2,5], logmu[2,6], logmu[2,7], logmu[2,8], logmu[2,9], logmu[2,10], logmu[2,11], logmu[2,12], logmu[2,13], logmu[2,14], logmu[2,15], logmu[2,16], logmu[2,17], logmu[2,18], logmu[2,19], logmu[2,20], logmu[2,21], logmu[3,1], logmu[3,2], logmu[3,3], 

In [11]:
loge[1]

Row,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,12.5806,12.0587,11.6203,11.2655,10.9942,10.8066,10.7024,10.6818,10.7289,10.7475,10.7217,10.6515,10.537,10.3899,10.2812,10.2226,10.2142,10.256,10.348,10.4901,10.6824
2,12.6849,12.1356,11.6767,11.3084,11.0305,10.843,10.7461,10.7396,10.8079,10.8571,10.8714,10.851,10.7958,10.7137,10.6524,10.62,10.6163,10.6413,10.6952,10.7778,10.8892
3,12.7288,12.1943,11.7589,11.4228,11.186,11.0483,11.0099,11.0708,11.2087,11.2908,11.2951,11.2214,11.0697,10.8578,10.6914,10.588,10.5479,10.5708,10.6568,10.806,11.0183
4,12.8136,12.4568,12.1458,11.8807,11.6614,11.4879,11.3603,11.2785,11.2341,11.1774,11.1,11.0019,10.8831,10.7523,10.662,10.621,10.6291,10.6866,10.7933,10.9492,11.1544
5,12.8056,12.3832,12.0304,11.7472,11.5336,11.3895,11.3151,11.3102,11.3622,11.3947,11.3948,11.3626,11.2981,11.2086,11.138,11.0937,11.0756,11.0838,11.1182,11.1789,11.2658
6,12.6959,12.3085,11.9849,11.7251,11.5291,11.3968,11.3284,11.3238,11.3734,11.4202,11.4546,11.4766,11.4863,11.4849,11.4809,11.4757,11.4692,11.4614,11.4523,11.442,11.4304


In [12]:
#Especification Model 1 (Univariate)
#Model 1 BLC individually by each group
modelesp1f1 = model1(b1, logef1, 5)
modelesp1f2 = model1(b2, logef2, 5)
modelesp1f3 = model1(b3, logef3, 5)
modelesp1f4 = model1(b4, logef4, 5)


#Chains model 1
chain_model1f1 = sample(modelesp1f1, NUTS(0.65), 10)
chain_model1f1 = sample(modelesp1f1, NUTS(0.65), 1000)
chain_model1f2 = sample(modelesp1f2, NUTS(0.65), 10)
chain_model1f2 = sample(modelesp1f2, NUTS(0.65), 1000)
chain_model1f3 = sample(modelesp1f3, NUTS(0.65), 10)
chain_model1f3 = sample(modelesp1f3, NUTS(0.65), 1000)
chain_model1f4 = sample(modelesp1f4, NUTS(0.65), 10)
chain_model1f4 = sample(modelesp1f4, NUTS(0.65), 1000)


#Serialize
serialize(dir * "Output/chain_model1f1.jls", chain_model1f1)
serialize(dir * "Output/chain_model1f2.jls", chain_model1f2)
serialize(dir * "Output/chain_model1f3.jls", chain_model1f3)
serialize(dir * "Output/chain_model1f4.jls", chain_model1f4)

┌ Info: Found initial step size
└   ϵ = 1.52587890625e-6
Sampling: 100%|█████████████████████████████████████████| Time: 0:00:00
┌ Info: Found initial step size
└   ϵ = 6.103515625e-6
Sampling: 100%|█████████████████████████████████████████| Time: 0:03:37
┌ Info: Found initial step size
└   ϵ = 4.8828125e-5
Sampling: 100%|█████████████████████████████████████████| Time: 0:00:00
┌ Info: Found initial step size
└   ϵ = 4.76837158203125e-8
Sampling: 100%|█████████████████████████████████████████| Time: 0:04:31
┌ Info: Found initial step size
└   ϵ = 5.960464477539063e-9
Sampling: 100%|█████████████████████████████████████████| Time: 0:00:00
┌ Info: Found initial step size
└   ϵ = 5.960464477539063e-9
Sampling: 100%|█████████████████████████████████████████| Time: 0:02:22
┌ Info: Found initial step size
└   ϵ = 1.52587890625e-6
┌ Info: Found initial step size
└   ϵ = 7.450580596923829e-10
Sampling: 100%|█████████████████████████████████████████| Time: 0:04:25
